In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import mlx .core as mx

from tqdm import tqdm


In [2]:
weather_db_directory =Path ('/Users/aaronspaulding/data/weather_db_hurricane_regions/')
outage_output_directory =weather_db_directory /'outage'
outage_output_directory .mkdir (parents =True ,exist_ok =True )

base_outage_data_directory =Path ('/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/data/Utilities/')

utilities_with_outage_information =[
'Orange and Rockland Utilities Inc',
'Consolidated Edison, Inc',
'Eversource',
'Dominion Energy',
'Public Service Electric and Gas',
'American Electric Power Texas',
]


In [4]:
relevant_storms =gpd .read_feather ('data_cache/relevant_storm_tracks.feather')
relevant_storms =relevant_storms [::-1 ].reset_index (drop =True )

relevant_storms [['SID','NAME','datetime_min','datetime_max']].head ()


,SID,NAME,datetime_min,datetime_max
0,2024279N21265,MILTON,2024-10-09 18:00:00+00:00,2024-10-10 15:00:00+00:00
1,2024268N17278,HELENE,2024-09-26 21:00:00+00:00,2024-09-28 18:00:00+00:00
2,2024259N31284,UNNAMED,2024-09-15 18:00:00+00:00,2024-09-17 00:00:00+00:00
3,2024253N21266,FRANCINE,2024-09-10 12:00:00+00:00,2024-09-14 00:00:00+00:00
4,2024225N14313,ERNESTO,2024-08-13 18:00:00+00:00,2024-08-14 15:00:00+00:00


In [5]:
h3_cells =gpd .read_feather (weather_db_directory /'h3_cells_cached.feather')
h3_cells =h3_cells [['index','geometry']].copy ().reset_index (drop =True )

if h3_cells .crs is None :
    h3_cells =h3_cells .set_crs ('EPSG:4326')

h3_index_to_row =pd .Series (np .arange (h3_cells .shape [0 ],dtype =np .int64 ),index =h3_cells ['index'])

h3_cells .shape


(3571572, 2)

In [6]:
all_outages =[]
for utility in utilities_with_outage_information :
    utility_outages_path =base_outage_data_directory /utility /'combined_outage_data'/'all_outages.feather'
    utility_outages =gpd .read_feather (utility_outages_path )
    utility_outages ['utility_name']=utility
    all_outages .append (utility_outages )

all_outages =pd .concat (all_outages ).reset_index (drop =True )
all_outages =gpd .GeoDataFrame (all_outages ,geometry ='geometry',crs ='EPSG:4326')

all_outages ['first_occurrence']=pd .to_datetime (all_outages ['first_occurrence'],utc =True )
all_outages =all_outages .dropna (subset =['first_occurrence','geometry']).reset_index (drop =True )
all_outages .shape


(486600, 7)

In [7]:
if h3_cells .crs !=all_outages .crs :
    h3_cells =h3_cells .to_crs (all_outages .crs )

all_outages ['hour']=all_outages ['first_occurrence'].dt .floor ('h').dt .tz_convert ('UTC').dt .tz_localize (None )

outages_with_cells =gpd .sjoin (
all_outages [['hour','geometry']],
h3_cells [['index','geometry']],
how ='inner',
predicate ='within',
)

hourly_outage_counts =(
outages_with_cells
.groupby (['hour','index'])
.size ()
.rename ('outage_count_for_this_hour')
.reset_index ()
)

hourly_outage_series =hourly_outage_counts .set_index (['hour','index'])['outage_count_for_this_hour']

hourly_outage_counts .head ()


,hour,index,outage_count_for_this_hour
0,2016-08-28 18:00:00,8826d901b1fffff,1
1,2016-08-28 18:00:00,8826d901b5fffff,1
2,2016-08-28 18:00:00,8826d90e45fffff,1
3,2016-08-28 18:00:00,8826d90f63fffff,1
4,2016-08-28 18:00:00,8826d94089fffff,1


In [8]:
def to_utc_naive (ts ):
    ts =pd .Timestamp (ts )
    if ts .tzinfo is None :
        return ts
    return ts .tz_convert ('UTC').tz_localize (None )

for j ,row in tqdm (relevant_storms .iterrows (),total =relevant_storms .shape [0 ]):
    storm_id =row ['SID']
    storm_name =row ['NAME']

    storm_start =to_utc_naive (row ['datetime_min']).floor ('h')
    storm_end =to_utc_naive (row ['datetime_max']).ceil ('h')+pd .to_timedelta (1 ,'d')

    dates =pd .date_range (storm_start ,storm_end ,freq ='h')
    dates =pd .DataFrame ({'datetime':dates })
    dates ['file_name']=dates ['datetime'].dt .strftime ('%Y_%m_%dT%H_%M_%S')+'.npy'
    dates ['file_exists']=dates ['file_name'].apply (lambda x :(outage_output_directory /x ).exists ())

    number_of_missing_files =int ((~dates ['file_exists']).sum ())
    relevant_storms .loc [j ,'number_of_missing_outage_files']=number_of_missing_files

    if number_of_missing_files ==0 :
        continue

    missing_dates =dates [~dates ['file_exists']].reset_index (drop =True )

    for _ ,row_ in tqdm (missing_dates .iterrows (),total =missing_dates .shape [0 ],leave =False ):
        dt =row_ ['datetime']
        file_name =row_ ['file_name']

        outage_data =np .zeros (h3_cells .shape [0 ],dtype =np .float32 )

        try :
            counts_this_hour =hourly_outage_series .loc [dt ]
        except KeyError :
            counts_this_hour =None

        if counts_this_hour is not None and len (counts_this_hour )>0 :
            cell_index_values =counts_this_hour .index .to_numpy ()
            row_positions =h3_index_to_row .loc [cell_index_values ].to_numpy (dtype =np .int64 )
            outage_data [row_positions ]=counts_this_hour .to_numpy (dtype =np .float32 )

        mx .save (outage_output_directory /file_name ,mx .array (outage_data ,dtype =mx .float16 ))


  0%|          | 0/46 [00:00<?, ?it/s]
                                      
  0%|          | 0/34 [00:00<?, ?it/s]
                                      
  0%|          | 0/40 [00:00<?, ?it/s]
                                      
  0%|          | 0/52 [00:00<?, ?it/s]
                                      
  0%|          | 0/49 [00:00<?, ?it/s]
                                      
  0%|          | 0/34 [00:00<?, ?it/s]
                                      
  0%|          | 0/49 [00:00<?, ?it/s]
                                      
  0%|          | 0/34 [00:00<?, ?it/s]
                                      
  0%|          | 0/46 [00:00<?, ?it/s]
                                      
  0%|          | 0/55 [00:00<?, ?it/s]
                                      
  0%|          | 0/40 [00:00<?, ?it/s]
                                      
  0%|          | 0/34 [00:00<?, ?it/s]
                                      
  0%|          | 0/43 [00:00<?, ?it/s]
                         

In [9]:
created_files =sorted (outage_output_directory .glob ('*.npy'))
print ('Outage files in output directory:',len (created_files ))

if len (created_files )>0 :
    sample =mx .load (str (created_files [0 ]))
    print ('Sample file:',created_files [0 ].name )
    print ('Sample shape:',sample .shape )
    print ('Sample dtype:',sample .dtype )
    print ('Sample max outage count:',float (mx .max (sample )))


Outage files in output directory: 7331
Sample file: 2014_06_28T18_00_00.npy
Sample shape: (3571572,)
Sample dtype: mlx.core.float16
Sample max outage count: 0.0
